# Training Proposed Model — IndoBERT + CNN 1D

**Notebook:** `03_indobert_cnn_training.ipynb`  
**Peneliti:** [Nama Peneliti]  
**Tanggal:** [Tanggal Pengerjaan]  

---

## Deskripsi

Notebook ini melatih **proposed model hybrid IndoBERT + CNN 1D** untuk analisis sentimen Program Makan Bergizi Gratis (MBG) menggunakan pendekatan **Staged Grid Search** dalam 3 phase:

| Phase | Tujuan | Runs | Kondisi Data |
|---|---|---|---|
| **Phase 1** | Cari BERT hyperparameter terbaik (CNN fixed default) | 60 | S1 |
| **Phase 2** | Cari CNN hyperparameter terbaik (BERT fixed dari Phase 1) | 80 | S1 |
| **Phase 3** | Validasi final + evaluasi test set | 10 | S1 + S2 |

---

## Cara Penggunaan Notebook Ini

Notebook ini **dijalankan dalam 3 sesi terpisah** — tidak bisa sekaligus dalam satu sesi karena Phase 2 bergantung pada hasil Phase 1, dan Phase 3 bergantung pada hasil Phase 2.

### Sesi 1 — Phase 1
1. Jalankan sel **0 hingga 4** (Setup, Konfigurasi, Data, Model, Utilities)
2. Jalankan sel **Phase 1 Training**
3. Tunggu selesai (~15–20 jam) — hasil tersimpan otomatis ke `phase1_results.csv`
4. Buka `phase1_results.csv`, cari baris dengan `f1_macro` tertinggi
5. **Update** `BEST_LR`, `BEST_BS`, `BEST_WD` di sel Konfigurasi

### Sesi 2 — Phase 2
6. Jalankan ulang sel **0 hingga 4**
7. Jalankan sel **Phase 2 Training**
8. Tunggu selesai (~20–25 jam) — hasil tersimpan ke `phase2_results.csv`
9. **Update** `BEST_NGRAM`, `BEST_FILTER`, `BEST_DROPOUT`, `BEST_ACTIVATION` di sel Konfigurasi

### Sesi 3 — Phase 3
10. Jalankan ulang sel **0 hingga 4**
11. Jalankan sel **Phase 3 Training + Evaluasi Final**
12. Model terbaik tersimpan ke folder `saved_models/`

---

## Arsitektur Proposed Model

```
Input Tweet (text_bert, max_len=128)
        ↓
IndoBERT-base-p2 → last_hidden_state [batch, 128, 768]
        ↓
CNN 1D Multi-Kernel (kernel per ngram_size)
  Conv1D(k1, F) → Activation → GlobalMaxPool → [batch, F]
  Conv1D(k2, F) → Activation → GlobalMaxPool → [batch, F]
  Conv1D(k3, F) → Activation → GlobalMaxPool → [batch, F]
        ↓
Concatenate → [batch, F×3] → Dropout
        ↓
Dense(256) → Activation → Dropout
        ↓
Dense(3) → Softmax [positive, negative, neutral]
```

**Referensi:**  
- Devlin et al. (2019). BERT. *NAACL-HLT 2019*.  
- Kim, Y. (2014). CNN for Sentence Classification. *EMNLP 2014*.  
- Wilie et al. (2020). IndoNLU. *AACL-IJCNLP 2020*.

## 0. Setup
**Jalankan di awal SETIAP sesi.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers torch scikit-learn -q

import os, warnings, random, itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Konfigurasi Global
**Jalankan di awal SETIAP sesi.**  
Sebelum Phase 2: update `BEST_LR`, `BEST_BS`, `BEST_WD`.  
Sebelum Phase 3: update semua `BEST_*`.

In [ ]:
BASE_DIR    = '/content/drive/My Drive/skripsi/dataset/mbg'
INPUT_PATH  = f'{BASE_DIR}/processed/mbg_labeled.csv'
OUTPUT_DIR  = f'{BASE_DIR}/model_results'
MODEL_DIR   = f'{BASE_DIR}/saved_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

BERT_MODEL  = 'indobenchmark/indobert-base-p2'
MAX_LEN     = 128
NUM_CLASSES = 3
LABEL2ID    = {'positive': 0, 'negative': 1, 'neutral': 2}
ID2LABEL    = {0: 'positive', 1: 'negative', 2: 'neutral'}

MAX_EPOCHS   = 10
PATIENCE     = 3
N_FOLDS      = 5
WARMUP_RATIO = 0.1
ADAM_EPS     = 1e-8

CNN_DEFAULT = {'ngram': [1,2,3], 'filter_size': 128, 'dropout': 0.3, 'activation': 'relu'}

PHASE1_GRID = {
    'lr'          : [1e-5, 2e-5, 3e-5],
    'batch_size'  : [16, 32],
    'weight_decay': [0.001, 0.01],
}
PHASE2_GRID = {
    'ngram'      : [[1,2,3], [2,3,4]],
    'filter_size': [128, 256],
    'dropout'    : [0.3, 0.5],
    'activation' : ['relu', 'gelu'],
}

# ── UPDATE SETELAH PHASE 1 SELESAI ───────────────────────────────────────────
BEST_LR = None   # contoh: 2e-5
BEST_BS = None   # contoh: 16
BEST_WD = None   # contoh: 0.01

# ── UPDATE SETELAH PHASE 2 SELESAI ───────────────────────────────────────────
BEST_NGRAM      = None   # contoh: [1,2,3]
BEST_FILTER     = None   # contoh: 128
BEST_DROPOUT    = None   # contoh: 0.3
BEST_ACTIVATION = None   # contoh: 'relu'

p1_total = len(list(itertools.product(*PHASE1_GRID.values())))
p2_total = len(list(itertools.product(*PHASE2_GRID.values())))
print(f'Phase 1: {p1_total} kombinasi × {N_FOLDS} fold = {p1_total*N_FOLDS} runs')
print(f'Phase 2: {p2_total} kombinasi × {N_FOLDS} fold = {p2_total*N_FOLDS} runs')
print(f'Phase 3: 2 kondisi × {N_FOLDS} fold = {2*N_FOLDS} runs + evaluasi final')

## 2. Load & Split Data
**Jalankan di awal SETIAP sesi.**  
Test set (20%) dipisah di awal dan disimpan agar konsisten antar sesi.

In [ ]:
df = pd.read_csv(INPUT_PATH, low_memory=False)
df = df[df['label'].isin(['positive','negative','neutral'])].copy()
df['label_id'] = df['label'].map(LABEL2ID)
df = df.dropna(subset=['text_bert','label_id']).reset_index(drop=True)

print(f'Total data berlabel: {len(df):,}')
for lbl, cnt in df['label'].value_counts().items():
    print(f'  {lbl:12s}: {cnt:,} ({cnt/len(df)*100:.1f}%)')

# Cek jika test set sudah pernah dibuat (konsistensi antar sesi)
test_set_path = f'{OUTPUT_DIR}/test_set_fixed.csv'
if os.path.exists(test_set_path):
    df_test     = pd.read_csv(test_set_path)
    test_ids    = set(df_test['id_str'].astype(str))
    df_trainval = df[~df['id_str'].astype(str).isin(test_ids)].reset_index(drop=True)
    df_test     = df[df['id_str'].astype(str).isin(test_ids)].reset_index(drop=True)
    print(f'\nTest set dimuat dari file existing (konsistensi antar sesi).')
else:
    _, _, idx_tv, idx_test = train_test_split(
        df['text_bert'].values, df['label_id'].values,
        test_size=0.2, random_state=SEED, stratify=df['label_id'].values
    )
    df_trainval = df.iloc[idx_tv].reset_index(drop=True)
    df_test     = df.iloc[idx_test].reset_index(drop=True)
    df_test.to_csv(test_set_path, index=False)
    print(f'\nTest set baru dibuat dan disimpan.')

print(f'Train+Val : {len(df_trainval):,} | Test (fixed): {len(df_test):,}')

## 3. Dataset & Arsitektur Model
**Jalankan di awal SETIAP sesi.**

In [ ]:
print(f'Memuat tokenizer: {BERT_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
print('Tokenizer berhasil dimuat.')


class MBGDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]), max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }


class IndoBERTCNN(nn.Module):
    """
    Proposed model: IndoBERT sebagai contextual feature extractor,
    diikuti CNN 1D multi-kernel untuk menangkap pola n-gram lokal.
    """
    def __init__(self, bert_model_name, num_classes,
                 ngram_sizes, filter_size, dropout, activation):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(bert_model_name)
        hidden       = self.bert.config.hidden_size  # 768
        act_fn       = nn.GELU() if activation == 'gelu' else nn.ReLU()

        self.convs   = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(hidden, filter_size, kernel_size=k, padding=k//2),
                act_fn
            ) for k in ngram_sizes
        ])
        self.drop1   = nn.Dropout(dropout)
        self.fc1     = nn.Linear(filter_size * len(ngram_sizes), 256)
        self.act_fc  = nn.GELU() if activation == 'gelu' else nn.ReLU()
        self.drop2   = nn.Dropout(dropout)
        self.fc2     = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):
        seq = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask).last_hidden_state
        x   = seq.permute(0, 2, 1)  # [batch, 768, 128]
        pooled = [F.adaptive_max_pool1d(conv(x), 1).squeeze(-1) for conv in self.convs]
        x   = torch.cat(pooled, dim=1)
        x   = self.drop1(x)
        x   = self.act_fc(self.fc1(x))
        x   = self.drop2(x)
        return self.fc2(x)


print('✅ Dataset dan arsitektur model berhasil didefinisikan.')

## 4. Training Utilities
**Jalankan di awal SETIAP sesi.**

In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        'accuracy'   : round(accuracy_score(y_true, y_pred), 4),
        'precision'  : round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'recall'     : round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'f1_macro'   : round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'f1_weighted': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 4),
    }

def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, preds_all, labels_all = 0, [], []
    for batch in loader:
        ids, mask, lbls = (batch['input_ids'].to(device),
                           batch['attention_mask'].to(device),
                           batch['label'].to(device))
        optimizer.zero_grad()
        logits = model(ids, mask)
        loss   = criterion(logits, lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
        labels_all.extend(lbls.cpu().numpy())
    return total_loss / len(loader), f1_score(labels_all, preds_all, average='macro', zero_division=0)

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, preds_all, labels_all = 0, [], []
    with torch.no_grad():
        for batch in loader:
            ids, mask, lbls = (batch['input_ids'].to(device),
                               batch['attention_mask'].to(device),
                               batch['label'].to(device))
            logits = model(ids, mask)
            total_loss += criterion(logits, lbls).item()
            preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
            labels_all.extend(lbls.cpu().numpy())
    return total_loss / len(loader), compute_metrics(labels_all, preds_all)

def apply_undersampling(X, y, seed=42):
    min_n = min(np.bincount(y))
    X_r, y_r = [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        s   = resample(idx, n_samples=min_n, random_state=seed, replace=False)
        X_r.append(X[s]); y_r.append(y[s])
    X_r, y_r = np.concatenate(X_r), np.concatenate(y_r)
    shuf = np.random.RandomState(seed).permutation(len(X_r))
    return X_r[shuf], y_r[shuf]

def build_criterion(y_train, condition, device):
    if condition == 'S2':
        return nn.CrossEntropyLoss()
    cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    return nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float).to(device))

def run_kfold(df_tv, lr, batch_size, weight_decay, ngram, filter_size,
              dropout, activation, data_condition='S1'):
    """
    Stratified K-Fold untuk satu kombinasi hyperparameter.
    data_condition: 'S1'=Imbalanced+ClassWeight, 'S2'=RandomUndersampling
    Returns: dict metrik rata-rata fold.
    """
    skf  = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    X_tv = df_tv['text_bert'].values
    y_tv = df_tv['label_id'].values
    fold_metrics = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tv, y_tv), 1):
        X_tr, y_tr = X_tv[tr_idx], y_tv[tr_idx]
        X_val, y_val = X_tv[val_idx], y_tv[val_idx]
        if data_condition == 'S2':
            X_tr, y_tr = apply_undersampling(X_tr, y_tr)

        criterion = build_criterion(y_tr, data_condition, DEVICE)
        train_dl  = DataLoader(MBGDataset(X_tr, y_tr, tokenizer, MAX_LEN),
                               batch_size=batch_size, shuffle=True,  num_workers=2)
        val_dl    = DataLoader(MBGDataset(X_val, y_val, tokenizer, MAX_LEN),
                               batch_size=batch_size, shuffle=False, num_workers=2)

        model = IndoBERTCNN(BERT_MODEL, NUM_CLASSES, ngram,
                            filter_size, dropout, activation).to(DEVICE)
        total_steps  = len(train_dl) * MAX_EPOCHS
        warmup_steps = int(total_steps * WARMUP_RATIO)
        optimizer = AdamW(model.parameters(), lr=lr,
                          weight_decay=weight_decay, eps=ADAM_EPS)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )

        best_loss, patience_cnt, best_m = float('inf'), 0, None
        for epoch in range(1, MAX_EPOCHS + 1):
            tr_loss, _  = train_epoch(model, train_dl, optimizer, scheduler, criterion, DEVICE)
            val_loss, m = eval_epoch(model, val_dl, criterion, DEVICE)
            if val_loss < best_loss:
                best_loss, best_m, patience_cnt = val_loss, m, 0
            else:
                patience_cnt += 1
                if patience_cnt >= PATIENCE: break

        fold_metrics.append(best_m)
        del model; torch.cuda.empty_cache()

    avg = {k: round(np.mean([m[k] for m in fold_metrics]), 4) for k in fold_metrics[0]}
    std = {f'{k}_std': round(np.std([m[k] for m in fold_metrics]), 4) for k in fold_metrics[0]}
    return {**avg, **std}

print('✅ Training utilities berhasil didefinisikan.')

---
# PHASE 1 — BERT Hyperparameter Search

Mencari kombinasi BERT hyperparameter terbaik dengan CNN fixed pada nilai default.

**CNN Default**: ngram=[1,2,3], filter=128, dropout=0.3, activation=ReLU  
**BERT Grid**: lr × batch_size × weight_decay = 3×2×2 = **12 kombinasi × 5 fold = 60 runs**  
**Kondisi data**: S1 (Imbalanced + Class Weighting)

> Estimasi waktu: **~15–20 jam** di GPU T4.

In [ ]:
p1_combos = list(itertools.product(
    PHASE1_GRID['lr'], PHASE1_GRID['batch_size'], PHASE1_GRID['weight_decay']
))
print(f'Total kombinasi Phase 1: {len(p1_combos)} ({len(p1_combos)*N_FOLDS} runs)')
for i, (lr, bs, wd) in enumerate(p1_combos, 1):
    print(f'  [{i:02d}] lr={lr:.0e} | bs={bs} | wd={wd}')

In [ ]:
p1_results = []
for i, (lr, bs, wd) in enumerate(p1_combos, 1):
    print(f'\n[{i:02d}/{len(p1_combos)}] lr={lr:.0e} | bs={bs} | wd={wd}')
    res = run_kfold(
        df_trainval, lr=lr, batch_size=bs, weight_decay=wd,
        ngram=CNN_DEFAULT['ngram'], filter_size=CNN_DEFAULT['filter_size'],
        dropout=CNN_DEFAULT['dropout'], activation=CNN_DEFAULT['activation'],
        data_condition='S1'
    )
    p1_results.append({'combo_id': i, 'lr': lr, 'batch_size': bs, 'weight_decay': wd, **res})
    print(f'  F1-Macro: {res["f1_macro"]:.4f} ± {res["f1_macro_std"]:.4f}')
    pd.DataFrame(p1_results).to_csv(f'{OUTPUT_DIR}/phase1_results.csv', index=False)

df_p1 = pd.DataFrame(p1_results).sort_values('f1_macro', ascending=False)
best  = df_p1.iloc[0]
print('\n' + '='*55)
print(' PHASE 1 RESULTS — Sorted by F1-Macro')
print('='*55)
print(df_p1[['combo_id','lr','batch_size','weight_decay','f1_macro','f1_macro_std','accuracy']].to_string(index=False))
print(f'\nBERT Config Terbaik: lr={best["lr"]} | bs={int(best["batch_size"])} | wd={best["weight_decay"]}')
print(f'F1-Macro: {best["f1_macro"]:.4f}')
print('\nUpdate BEST_LR, BEST_BS, BEST_WD di sel Konfigurasi sebelum Phase 2!')

In [ ]:
df_p1_v = pd.DataFrame(p1_results).sort_values('f1_macro', ascending=False).reset_index(drop=True)
df_p1_v['config'] = df_p1_v.apply(lambda r: f"lr={r['lr']:.0e}\nbs={int(r['batch_size'])}\nwd={r['weight_decay']}", axis=1)
colors = ['#1D9E75'] + ['#534AB7'] * (len(df_p1_v) - 1)

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(range(len(df_p1_v)), df_p1_v['f1_macro'], color=colors, edgecolor='none', alpha=0.9)
ax.errorbar(range(len(df_p1_v)), df_p1_v['f1_macro'],
            yerr=df_p1_v['f1_macro_std'], fmt='none', ecolor='#888780', capsize=4)
ax.set_xticks(range(len(df_p1_v)))
ax.set_xticklabels(df_p1_v['config'], fontsize=8)
ax.set_ylabel('F1-Macro (mean 5-fold)', fontsize=11)
ax.set_title('Phase 1 — BERT Hyperparameter Search', fontsize=12)
ax.set_ylim(df_p1_v['f1_macro'].min() - 0.05, df_p1_v['f1_macro'].max() + 0.05)
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/phase1_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
# PHASE 2 — CNN Hyperparameter Search

**Prasyarat**: `BEST_LR`, `BEST_BS`, `BEST_WD` sudah diupdate di sel Konfigurasi.

**BERT Config**: fixed dari Phase 1  
**CNN Grid**: ngram × filter × dropout × activation = 2×2×2×2 = **16 kombinasi × 5 fold = 80 runs**  
**Kondisi data**: S1 (Imbalanced + Class Weighting)

> Estimasi waktu: **~20–25 jam** di GPU T4.

In [ ]:
assert BEST_LR is not None, 'Update BEST_LR di sel Konfigurasi terlebih dahulu!'
assert BEST_BS is not None, 'Update BEST_BS di sel Konfigurasi terlebih dahulu!'
assert BEST_WD is not None, 'Update BEST_WD di sel Konfigurasi terlebih dahulu!'

p2_combos = list(itertools.product(
    PHASE2_GRID['ngram'], PHASE2_GRID['filter_size'],
    PHASE2_GRID['dropout'], PHASE2_GRID['activation']
))
print(f'BERT Config (fixed): lr={BEST_LR} | bs={BEST_BS} | wd={BEST_WD}')
print(f'Total kombinasi Phase 2: {len(p2_combos)} ({len(p2_combos)*N_FOLDS} runs)')
for i, (ng, fs, dr, ac) in enumerate(p2_combos, 1):
    print(f'  [{i:02d}] ngram={ng} | filter={fs} | dropout={dr} | act={ac}')

In [ ]:
p2_results = []
for i, (ng, fs, dr, ac) in enumerate(p2_combos, 1):
    print(f'\n[{i:02d}/{len(p2_combos)}] ngram={ng} | filter={fs} | dropout={dr} | act={ac}')
    res = run_kfold(
        df_trainval, lr=BEST_LR, batch_size=BEST_BS, weight_decay=BEST_WD,
        ngram=ng, filter_size=fs, dropout=dr, activation=ac,
        data_condition='S1'
    )
    p2_results.append({'combo_id': i, 'ngram': str(ng), 'filter_size': fs,
                       'dropout': dr, 'activation': ac, **res})
    print(f'  F1-Macro: {res["f1_macro"]:.4f} ± {res["f1_macro_std"]:.4f}')
    pd.DataFrame(p2_results).to_csv(f'{OUTPUT_DIR}/phase2_results.csv', index=False)

df_p2 = pd.DataFrame(p2_results).sort_values('f1_macro', ascending=False)
best2 = df_p2.iloc[0]
print('\n' + '='*60)
print(' PHASE 2 RESULTS — Sorted by F1-Macro')
print('='*60)
print(df_p2[['combo_id','ngram','filter_size','dropout','activation',
             'f1_macro','f1_macro_std','accuracy']].to_string(index=False))
print(f'\nCNN Config Terbaik: ngram={best2["ngram"]} | filter={int(best2["filter_size"])} | dropout={best2["dropout"]} | act={best2["activation"]}')
print(f'F1-Macro: {best2["f1_macro"]:.4f}')
print('\nUpdate BEST_NGRAM, BEST_FILTER, BEST_DROPOUT, BEST_ACTIVATION sebelum Phase 3!')

In [ ]:
df_p2_v = pd.DataFrame(p2_results).sort_values('f1_macro', ascending=False).reset_index(drop=True)
df_p2_v['config'] = df_p2_v.apply(
    lambda r: f"ng={r['ngram']}\nf={int(r['filter_size'])}\ndr={r['dropout']}\n{r['activation']}", axis=1)
colors = ['#1D9E75'] + ['#534AB7'] * (len(df_p2_v) - 1)

fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(range(len(df_p2_v)), df_p2_v['f1_macro'], color=colors, edgecolor='none', alpha=0.9)
ax.errorbar(range(len(df_p2_v)), df_p2_v['f1_macro'],
            yerr=df_p2_v['f1_macro_std'], fmt='none', ecolor='#888780', capsize=4)
ax.set_xticks(range(len(df_p2_v)))
ax.set_xticklabels(df_p2_v['config'], fontsize=7)
ax.set_ylabel('F1-Macro (mean 5-fold)', fontsize=11)
ax.set_title('Phase 2 — CNN Hyperparameter Search', fontsize=12)
ax.set_ylim(df_p2_v['f1_macro'].min() - 0.05, df_p2_v['f1_macro'].max() + 0.05)
sns.despine(); plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/phase2_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
# PHASE 3 — Validasi Final & Evaluasi Test Set

**Prasyarat**: Semua `BEST_*` sudah diupdate di sel Konfigurasi.

Menjalankan konfigurasi terbaik pada dua kondisi data (S1 dan S2), kemudian melatih model final pada seluruh train+val set dan mengevaluasi pada **test set fixed (1.328 data)** yang belum pernah dilihat model sebelumnya.

> Test set hanya digunakan **satu kali** di tahap ini — tidak boleh digunakan untuk keputusan apapun sebelumnya.

In [ ]:
assert BEST_LR         is not None, 'BEST_LR belum diisi!'
assert BEST_BS         is not None, 'BEST_BS belum diisi!'
assert BEST_WD         is not None, 'BEST_WD belum diisi!'
assert BEST_NGRAM      is not None, 'BEST_NGRAM belum diisi!'
assert BEST_FILTER     is not None, 'BEST_FILTER belum diisi!'
assert BEST_DROPOUT    is not None, 'BEST_DROPOUT belum diisi!'
assert BEST_ACTIVATION is not None, 'BEST_ACTIVATION belum diisi!'

print('Konfigurasi Final Proposed Model:')
print(f'  BERT : lr={BEST_LR} | bs={BEST_BS} | wd={BEST_WD}')
print(f'  CNN  : ngram={BEST_NGRAM} | filter={BEST_FILTER} | dropout={BEST_DROPOUT} | act={BEST_ACTIVATION}')
print(f'\nPhase 3: S1 (5 fold) + S2 (5 fold) + evaluasi final test set')

In [ ]:
p3_results = []
for cond, desc in [('S1', 'Imbalanced + Class Weighting'), ('S2', 'Random Undersampling')]:
    print(f'\nTraining {cond} ({desc})...')
    res = run_kfold(
        df_trainval, lr=BEST_LR, batch_size=BEST_BS, weight_decay=BEST_WD,
        ngram=BEST_NGRAM, filter_size=BEST_FILTER,
        dropout=BEST_DROPOUT, activation=BEST_ACTIVATION,
        data_condition=cond
    )
    p3_results.append({'condition': cond, 'description': desc, **res})
    print(f'  F1-Macro (val): {res["f1_macro"]:.4f} ± {res["f1_macro_std"]:.4f}')

df_p3 = pd.DataFrame(p3_results)
df_p3.to_csv(f'{OUTPUT_DIR}/phase3_kfold_results.csv', index=False)
print('\n' + '='*55)
print(' PHASE 3 — K-FOLD VALIDATION RESULTS')
print('='*55)
print(df_p3[['condition','description','f1_macro','f1_macro_std','accuracy']].to_string(index=False))

best_cond = df_p3.sort_values('f1_macro', ascending=False).iloc[0]['condition']
print(f'\nKondisi terbaik untuk evaluasi final: {best_cond}')

In [ ]:
print(f'Training final model (kondisi {best_cond}) pada seluruh train+val...')

X_full = df_trainval['text_bert'].values
y_full = df_trainval['label_id'].values
if best_cond == 'S2':
    X_full, y_full = apply_undersampling(X_full, y_full)

criterion_f = build_criterion(y_full, best_cond, DEVICE)
train_dl_f  = DataLoader(MBGDataset(X_full, y_full, tokenizer, MAX_LEN),
                         batch_size=BEST_BS, shuffle=True,  num_workers=2)
test_dl_f   = DataLoader(MBGDataset(df_test['text_bert'].values,
                                    df_test['label_id'].values, tokenizer, MAX_LEN),
                         batch_size=BEST_BS, shuffle=False, num_workers=2)

model_f = IndoBERTCNN(BERT_MODEL, NUM_CLASSES, BEST_NGRAM,
                      BEST_FILTER, BEST_DROPOUT, BEST_ACTIVATION).to(DEVICE)
total_steps  = len(train_dl_f) * MAX_EPOCHS
optimizer_f  = AdamW(model_f.parameters(), lr=BEST_LR, weight_decay=BEST_WD, eps=ADAM_EPS)
scheduler_f  = get_linear_schedule_with_warmup(
    optimizer_f, int(total_steps * WARMUP_RATIO), total_steps)

best_loss, patience_cnt, best_state = float('inf'), 0, None
for epoch in range(1, MAX_EPOCHS + 1):
    tr_loss, _  = train_epoch(model_f, train_dl_f, optimizer_f, scheduler_f, criterion_f, DEVICE)
    val_loss, m = eval_epoch(model_f, test_dl_f, criterion_f, DEVICE)
    print(f'  Epoch {epoch:02d} | tr_loss={tr_loss:.4f} | val_loss={val_loss:.4f} | F1={m["f1_macro"]:.4f}')
    if val_loss < best_loss:
        best_loss, best_state, patience_cnt = val_loss, {k: v.clone() for k,v in model_f.state_dict().items()}, 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f'  Early stopping di epoch {epoch}'); break

model_f.load_state_dict(best_state)

# ── Evaluasi final ────────────────────────────────────────────────────────────
_, final_metrics = eval_epoch(model_f, test_dl_f, criterion_f, DEVICE)

model_f.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_dl_f:
        logits = model_f(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        all_labels.extend(batch['label'].numpy())

label_names = ['positive', 'negative', 'neutral']
cm = confusion_matrix(all_labels, all_preds)

print('\n' + '='*55)
print(f' EVALUASI FINAL — TEST SET (kondisi {best_cond})')
print('='*55)
for k, v in final_metrics.items(): print(f'  {k:15s}: {v:.4f}')
print(f'\n{classification_report(all_labels, all_preds, target_names=label_names)}')

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel('Predicted', fontsize=11); ax.set_ylabel('Actual', fontsize=11)
ax.set_title(f'Confusion Matrix — IndoBERT+CNN ({best_cond})\nF1-Macro: {final_metrics["f1_macro"]:.4f}', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/cm_proposed_{best_cond}.png', dpi=150, bbox_inches='tight')
plt.show()

# Simpan model
torch.save({
    'model_state_dict': model_f.state_dict(),
    'bert_config'     : {'lr': BEST_LR, 'batch_size': BEST_BS, 'weight_decay': BEST_WD},
    'cnn_config'      : {'ngram': BEST_NGRAM, 'filter': BEST_FILTER,
                         'dropout': BEST_DROPOUT, 'activation': BEST_ACTIVATION},
    'data_condition'  : best_cond,
    'test_metrics'    : final_metrics,
    'label2id'        : LABEL2ID, 'id2label': ID2LABEL,
}, f'{MODEL_DIR}/indobert_cnn_final_{best_cond}.pt')

print(f'\nModel disimpan ke: {MODEL_DIR}/indobert_cnn_final_{best_cond}.pt')
print('Langkah selanjutnya: Notebook 04 — BERT-only Baseline')
print(f'Gunakan BERT config: lr={BEST_LR} | bs={BEST_BS} | wd={BEST_WD}')

---

## Catatan Metodologis untuk Laporan

### Staged Grid Search
Pencarian hyperparameter dilakukan secara bertahap untuk menghindari *combinatorial explosion*. Full grid search pada semua parameter secara bersamaan menghasilkan 192 kombinasi × 5 fold = 960 runs — tidak feasible. Pendekatan staged mengurangi total runs menjadi ~150 dengan asumsi bahwa hyperparameter BERT dan CNN memberikan pengaruh yang relatif independen terhadap performa model.

### Early Stopping
Early stopping dengan patience=3 berbasis validation loss mencegah overfitting dan mengefisienkan komputasi. Model terbaik per fold dikembalikan berdasarkan validation loss terendah, bukan epoch terakhir.

### Penanganan Imbalanced Data
Dua strategi dibandingkan: (1) class weighting pada loss function; (2) random undersampling ke kelas terkecil (1.937 sampel/kelas). Strategi terbaik dipilih berdasarkan F1-Macro tertinggi pada K-Fold validation.

### Evaluasi
Test set (20% = 1.328 data) dipisah sebelum seluruh proses training dan hanya digunakan sekali pada evaluasi final. Metrik utama adalah F1-Macro karena bersifat robust terhadap class imbalance dengan memberikan bobot equal per kelas.